# Data and Sampling Distributions

## Importing Libraries and Cleaned Dataset

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import zscore

# All figures from this notebook are saved here (created if missing).
FIG_DIR = '../figures/02_data_and_sampling_distributions'
os.makedirs(FIG_DIR, exist_ok=True)

df = pd.read_csv('../data/cleaned_data.csv', keep_default_na=False)
df.head()

In [ ]:
df.info()

## Sampling Distribution

### Central Limit Theorem

I took three samples (a sample of 1000 values, a sample of 1,000 means of 5 values, a sample of 1,000 means of 30 values and a sample of 1,000 means of 100 values) from the `SalePrice` column of the dataset and plotted a histogram of each sample.

In [ ]:
sample_data = pd.DataFrame({
'Sale Price': df['SalePrice'].sample(1000),
'type': 'Data',
})
sample_mean_05 = pd.DataFrame({
'Sale Price': [df['SalePrice'].sample(5).mean() for _ in range(1000)],
'type': 'Mean of 5',
})
sample_mean_30 = pd.DataFrame({
'Sale Price': [df['SalePrice'].sample(30).mean() for _ in range(1000)],
'type': 'Mean of 30',
})
sample_mean_100 = pd.DataFrame({
'Sale Price': [df['SalePrice'].sample(100).mean() for _ in range(1000)],
'type': 'Mean of 100',
})
results = pd.concat([sample_data, sample_mean_05, sample_mean_30, sample_mean_100])

g = sns.FacetGrid(results, col='type', col_wrap=1, height=2, aspect=4, sharex=True, sharey=False)
g.map(plt.hist, 'Sale Price', bins=40)

g.tight_layout()
g.savefig(f'{FIG_DIR}/SamplingDistribution.png', dpi=600, bbox_inches="tight")

The figure illustrates the Central Limit Theorem correctly:
* Data: The original sale prices are strongly right-skewed with a long upper tail.
* Mean of 5: The sampling distribution of the mean is less skewed and has a smaller spread.
* Mean of 30: The distribution is approximately normal.
* Mean of 100: The distribution is very close to a normal distribution and is tightly concentrated around the population mean.

### Standard Error

The standard error is the variability (standard deviation) of a sample statistic over many samples. The standard error can be estimated using a statistic based on the standard deviation *s* of the sample values, and the sample size *n*:

$SE = \frac{s}{\sqrt{n}}$

In [ ]:
# Empirical SE
empirical_se = (
    results[results['type'] != 'Data']
    .groupby('type')['Sale Price']
    .std()
)

# Theoretical SE
sigma = df['SalePrice'].std(ddof=1)

theoretical = {
    5: sigma / np.sqrt(5),
    30: sigma / np.sqrt(30),
    100: sigma / np.sqrt(100)
}

# Comparison DataFrame
comparison = pd.DataFrame({
    'Sample Size': [5, 30, 100],
    'Empirical SE': [
        sample_mean_05['Sale Price'].std(ddof=1),
        sample_mean_30['Sale Price'].std(ddof=1),
        sample_mean_100['Sale Price'].std(ddof=1)
    ],
    'Theoretical SE': [
        sigma / np.sqrt(5),
        sigma / np.sqrt(30),
        sigma / np.sqrt(100)
    ]
})

# Long format for plotting
comparison_long = comparison.melt(
    id_vars='Sample Size',
    var_name='Type',
    value_name='Standard Error'
)

plt.plot(
    comparison['Sample Size'],
    comparison['Empirical SE'],
    marker='o',
    linewidth=2,
    label='Empirical'
)

plt.plot(
    comparison['Sample Size'],
    comparison['Theoretical SE'],
    marker='s',
    linewidth=2,
    linestyle='--',
    label='Theoretical'
)

plt.xticks([5, 30, 60, 100])
plt.xlabel('Sample Size')
plt.ylabel('Standard Error')
plt.title('Empirical vs Theoretical Standard Error')
plt.legend()

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/EmpiricalvsTheoreticalSE.png', dpi=600, bbox_inches='tight')
plt.show()

* The empirical SE (blue) is very close to the theoretical SE (orange) for all sample sizes (5, 30, and 100). This indicates that your simulation accurately reproduces the theoretical sampling distribution of the sample mean.
* As the sample size increases, both empirical and theoretical SE decrease substantially. This confirms that larger samples produce more precise estimates of the population mean.
* The slight differences between the two curves are due to Monte Carlo sampling error. Since each empirical SE is estimated from only 1,000 simulated samples, it will not exactly equal the theoretical value. Increasing the number of simulations (e.g., to 10,000 or 100,000) would make the empirical curve converge even more closely to the theoretical one.

## Bootstrap Analysis

### Boostrap statistics of `median` `SalePrice`

In [ ]:
sale_prices = df['SalePrice'].to_numpy()

# Parameters
R = 10000  # Number of bootstrap samples
n = len(sale_prices)  # Sample size

# Original statistic (median)
original_median = np.median(sale_prices)

# Bootstrap medians
bootstrap_medians = np.empty(R)

for i in range(R):
    sample = np.random.choice(sale_prices, size=n, replace=True)
    bootstrap_medians[i] = np.median(sample)

# Bootstrap statistics
bootstrap_mean = np.mean(bootstrap_medians) # average of the bootstrap medians
bias = bootstrap_mean - original_median
standard_error = np.std(bootstrap_medians, ddof=1)

# 95% bootstrap confidence interval (percentile method)
ci_lower = np.percentile(bootstrap_medians, 2.5)
ci_upper = np.percentile(bootstrap_medians, 97.5)

print(f"Original median: {original_median:.2f}")
print(f"Mean of bootstrap medians: {bootstrap_mean:.2f}")
print(f"Bias: {bias:.2f}")
print(f"Standard error: {standard_error:.2f}")
print(f"95% CI: ({ci_lower:.2f}, {ci_upper:.2f})")

Bootstrap analysis of the SalePrice median shows that the original median sale price is 163,000. The bootstrap estimate of the median is 162939.99, resulting in a small bias of -60.01. Since this bias is negligible relative to the median (<0.1%), the sample median appears to be an unbiased estimate of the population median. The bootstrap standard error is 2,368.94, indicating that the estimated median would typically vary by about 2,369 across repeated samples. The 95% bootstrap confidence interval ranges from 158,900 to 167,700, suggesting that the true population median sale price is likely to lie within this interval. Overall, the bootstrap results indicate that the sample median is a stable and reliable estimate of the population median.

### Bootstrap distribution of `median` `SalePrice`

In [ ]:
plt.hist(bootstrap_medians, bins=30, edgecolor='white')

plt.axvline(original_median, color='red', linestyle='--', label='Original Median')
plt.axvline(ci_lower, color='green', linestyle=':', linewidth=1.5, label=f'CI Lower = {ci_lower:.2f}')
plt.axvline(ci_upper, color='green', linestyle=':', linewidth=1.5, label=f'CI Upper = {ci_upper:.2f}')

plt.xlabel('Bootstrap Median')
plt.ylabel('Frequency')
plt.title('Bootstrap Distribution of SalePrice Median')
plt.legend()
plt.savefig(f'{FIG_DIR}/BootstrapDistributionOfSalePriceMedian.png', dpi=600, bbox_inches='tight')
plt.show()

The bootstrap distribution of the median sale price is centred around the original sample median of 163,000, indicating that the bootstrap estimates are unbiased. The distribution is relatively narrow, suggesting low-sampling variability. Although the histogram is not perfectly bell-shaped and exhibits several distinct peaks, this is expected because the median is a discrete statistic and many bootstrap samples produce identical median values. The distribution nevertheless demonstrates that the sample median is a stable estimate of the population median.

## Standard Normal Distribution and QQ-plot of `SalePrice`

In [ ]:
saleprice_z = zscore(df['SalePrice'])

print(f"Standardized Mean: {saleprice_z.mean():.2f}")
print(f"Standardized Standard deviation: {saleprice_z.std():.2f}")

In [ ]:
plt.hist(saleprice_z, bins=30, edgecolor='white')
plt.xlabel("z-score")
plt.ylabel("Frequency")
plt.title("Standardized SalePrice")
plt.savefig(f'{FIG_DIR}/StandardizedSalePriceHist.png', dpi=600, bbox_inches='tight')
plt.show()

### QQ-Plot

In [ ]:
stats.probplot(df['SalePrice'], dist="norm", plot=plt)
plt.title("QQ Plot of SalePrice")
plt.savefig(f'{FIG_DIR}/SalePriceQQplot', dpi=600, bbox_inches='tight')
plt.show()

The QQ plot confirms that `SalePrice` is not normally distributed, which is consistent with the findings from Notebook 01. Most of the points in the middle lie close to the reference line, while the upper quantiles deviate noticeably above it. This is because the dataset contains a small number of houses with very high sale prices, resulting in a right-skewed distribution with a heavy upper tail. The lower tail shows only slight deviation from normality.

### Log Transformed QQ-Plot

In [ ]:
log_price = np.log1p(df['SalePrice'])

stats.probplot(log_price, dist='norm', plot=plt)
plt.title("QQ Plot of log(SalePrice)")
plt.savefig(f'{FIG_DIR}/LogSalePriceQQplot', dpi=600, bbox_inches='tight')
plt.show()

The QQ plot of `log(SalePrice)` shows that the data follow the reference line much more closely than the original `SalePrice` distribution. Although there are still slight deviations at the lower and upper tails, the overall pattern is nearly linear, indicating that the log transformation has substantially reduced the positive skewness. As a result, `log(SalePrice)` provides a much better approximation to a normal distribution and is more suitable for analyses that assume normality.